In [0]:
customers = spark.read.csv(
    "/FileStore/tables/customers_clean.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    "/FileStore/tables/products_clean.csv",
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    "/FileStore/tables/orders_clean.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    "/FileStore/tables/order_items_clean.csv",
    header=True,
    inferSchema=True
)

In [0]:
customers = spark.table("customers")
products = spark.table("products")
orders = spark.table("orders")
order_items = spark.table("order_items")

In [0]:
%sql
SELECT
date_trunc('month', order_date) AS month,
ROUND(SUM(quantity * unit_price), 2) AS revenue
FROM orders_clean o
JOIN order_items_clean oi
ON o.order_id = oi.order_id
GROUP BY 1
ORDER BY 1;

month,revenue
2024-06-01T00:00:00.000Z,1328665.61
2024-07-01T00:00:00.000Z,3729082.07
2024-08-01T00:00:00.000Z,4086270.75
2024-09-01T00:00:00.000Z,4251651.1
2024-10-01T00:00:00.000Z,4627728.76
2024-11-01T00:00:00.000Z,4516070.58
2024-12-01T00:00:00.000Z,4334369.9
2025-01-01T00:00:00.000Z,3878787.55
2025-02-01T00:00:00.000Z,3903921.01
2025-03-01T00:00:00.000Z,4201375.34


In [0]:
%sql
SELECT
date_trunc('month', order_date) AS month,

ROUND(
    SUM(quantity * unit_price),
    2
) AS revenue

FROM orders_clean o

JOIN order_items_clean oi
ON o.order_id = oi.order_id

GROUP BY 1

ORDER BY 1;

month,revenue
2024-06-01T00:00:00.000Z,1328665.61
2024-07-01T00:00:00.000Z,3729082.07
2024-08-01T00:00:00.000Z,4086270.75
2024-09-01T00:00:00.000Z,4251651.1
2024-10-01T00:00:00.000Z,4627728.76
2024-11-01T00:00:00.000Z,4516070.58
2024-12-01T00:00:00.000Z,4334369.9
2025-01-01T00:00:00.000Z,3878787.55
2025-02-01T00:00:00.000Z,3903921.01
2025-03-01T00:00:00.000Z,4201375.34


In [0]:
%sql
SELECT

c.customer_id,
c.name,

ROUND(
    SUM(quantity * unit_price),
    2
) AS total_spend

FROM customers_clean c

JOIN orders_clean o
ON c.customer_id = o.customer_id

JOIN order_items_clean oi
ON o.order_id = oi.order_id

GROUP BY
c.customer_id,
c.name

ORDER BY total_spend DESC

LIMIT 10;

customer_id,name,total_spend
199,Rebecca Colon,334531.49
369,Laura Malone,299309.26
647,Thomas Vargas,290331.03
163,Lori Rodriguez,286304.09
67,John Fletcher,283962.74
592,Karen Solomon,282705.2
947,Kimberly Collins,278632.88
936,Ronald Yoder,277807.98
397,Paul Herrera,271290.36
952,Michelle Downs,266693.37


In [0]:
%sql
SELECT

p.product_name,

ROUND(
    SUM(quantity * unit_price),
    2
) AS revenue

FROM products_clean p

JOIN order_items_clean oi
ON p.product_id = oi.product_id

GROUP BY p.product_name

ORDER BY revenue DESC

LIMIT 10;

product_name,revenue
Product_168,1347580.07
Product_12,1298260.8
Product_81,1255164.12
Product_100,1242003.42
Product_84,1212764.16
Product_167,1205288.26
Product_192,1189132.8
Product_190,1179820.92
Product_93,1135411.77
Product_25,1130582.4


In [0]:
%sql
SELECT

p.category,

ROUND(
    SUM(quantity * unit_price),
    2
) AS category_revenue

FROM products_clean p

JOIN order_items_clean oi
ON p.product_id = oi.product_id

GROUP BY p.category

ORDER BY category_revenue DESC;

category,category_revenue
Home,3.394507994E7
Fashion,2.91974144E7
Sports,2.22699869E7
Electronics,1.996513741E7


In [0]:
%sql
SELECT

customer_id,

ROUND(
SUM(quantity * unit_price),
2
) AS lifetime_value

FROM orders_clean o

JOIN order_items_clean oi
ON o.order_id = oi.order_id

GROUP BY customer_id

ORDER BY lifetime_value DESC;

customer_id,lifetime_value
199,334531.49
369,299309.26
647,290331.03
163,286304.09
67,283962.74
592,282705.2
947,278632.88
936,277807.98
397,271290.36
952,266693.37


In [0]:
%sql
SELECT

date_trunc(
'month',
signup_date
) AS signup_month,

COUNT(*) AS customers

FROM customers_clean

GROUP BY 1

ORDER BY 1;

signup_month,customers
null,195
2023-06-01T00:00:00.000Z,4
2023-07-01T00:00:00.000Z,30
2023-08-01T00:00:00.000Z,25
2023-09-01T00:00:00.000Z,22
2023-10-01T00:00:00.000Z,28
2023-11-01T00:00:00.000Z,23
2023-12-01T00:00:00.000Z,35
2024-01-01T00:00:00.000Z,31
2024-02-01T00:00:00.000Z,21


In [0]:
# spark.sql("""
# WITH cohort AS (
#   SELECT
#     customer_id,
#     date_trunc('month', MIN(order_date)) AS cohort_month
#   FROM orders_clean
#   GROUP BY customer_id
# ),
# activity AS (
#   SELECT
#     customer_id,
#     date_trunc('month', order_date) AS activity_month
#   FROM orders_clean
# )
# SELECT
#   c.cohort_month,
#   a.activity_month,
#   COUNT(DISTINCT a.customer_id) AS active_customers
# FROM cohort c
# JOIN activity a
#   ON c.customer_id = a.customer_id
# GROUP BY
#   c.cohort_month,
#   a.activity_month
# ORDER BY
#   c.cohort_month,
#   a.activity_month
# """)

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,customers_clean,false
default,order_items_clean,false
default,orders_clean,false
default,products_clean,false
